**Imports**

In [ ]:
from src.data.load_dataset import load_paired_data
from src.modules.super_resolution import SuperResolutionModel
import numpy as np
import torch
import matplotlib.pyplot as plt

**Load Dataset**

In [5]:
train_loader, test_loader, val_loader = load_paired_data()

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/45 [00:00<?, ?it/s]

**Super Resolution**

In [ ]:
FORCE_LEARNING = False
LOAD_BEST = True

sr_model = SuperResolutionModel(
    latent_dim=768, 
    input_channels=3, 
    learning_rate=0.001,
    image_size=256,    
    load_best=LOAD_BEST  
) 


if not sr_model.model_loaded or FORCE_LEARNING:
    history = sr_model.fit(
        train_loader=train_loader, 
        val_loader=val_loader,     
        epochs=50
    )

Blad podczas wczytywania modelu: Error(s) in loading state_dict for Encoder:
	size mismatch for conv1.weight: copying a param with shape torch.Size([64, 4, 4, 4]) from checkpoint, the shape in current model is torch.Size([64, 3, 4, 4]).
Model zostanie wytrenowany od nowa

Epoka 1/50


Training SR:   0%|          | 1/446 [00:06<48:43,  6.57s/it]

**Wyniki 256 -> 512 -> 1024**

In [ ]:
def show_sr_results(model, dataloader, num_samples=3):
    model.eval()
    device = model.device
    
    lr_imgs, _ = next(iter(dataloader))
    
    indices = torch.randperm(len(lr_imgs))[:num_samples]
    
    _, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            img_256 = lr_imgs[idx].unsqueeze(0).to(device)
            img_512, _ = model(img_256)
            img_1024, _ = model(img_512)
            imgs = [img_256, img_512, img_1024]
            titles = [f"Input (256x256)", f"Upscaled x2 (512x512)", f"Upscaled x4 (1024x1024)"]
            
            for j, img_tensor in enumerate(imgs):
                img_np = img_tensor.squeeze().cpu().permute(1, 2, 0).clamp(0, 1).numpy()
                
                ax = axes[i, j]
                ax.imshow(img_np)
                ax.set_title(titles[j])
                ax.axis('off')

    plt.tight_layout()
    plt.show()

show_sr_results(sr_model, test_loader, num_samples=3)

**Historia uczenia**

In [ ]:
print("=" * 60)
print("PODSUMOWANIE TRENINGU")
print("=" * 60)

print(f"\nLiczba epok: {len(history['train_loss'])}")
print(f"\nNajlepsza Train Loss: {min(history['train_loss']):.6f}")
print(f"Najlepsza Val Loss: {min(history['val_loss']):.6f}")
print(f"\nOstatnia Train Loss: {history['train_loss'][-1]:.6f}")
print(f"Ostatnia Val Loss: {history['val_loss'][-1]:.6f}")

train_improvement = ((history['train_loss'][0] - history['train_loss'][-1]) / history['train_loss'][0]) * 100
val_improvement = ((history['val_loss'][0] - history['val_loss'][-1]) / history['val_loss'][0]) * 100

print(f"\nPoprawa Train Loss: {train_improvement:.2f}%")
print(f"Poprawa Val Loss: {val_improvement:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoka')
axes[0].set_ylabel('Loss')
axes[0].set_title('Krzywa uczenia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

epochs = np.arange(1, len(history['train_loss']) + 1)
axes[1].plot(epochs, history['train_recon_loss'], label='Train Recon Loss', marker='o')
axes[1].plot(epochs, history['val_recon_loss'], label='Val Recon Loss', marker='s')
axes[1].set_xlabel('Epoka')
axes[1].set_ylabel('Reconstruction Loss')
axes[1].set_title('Strata rekonstrukcji')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()